# 03 — Model Training and Comparison

### Notebook roadmap
1. Setup and data loading
2. Evaluation protocol
3. Baseline reference
4. Candidate models (presentation of each)
5. Training the models
6. Cross-validated comparison
7. Detailed evaluation on a held-out part of the training data
8. Error analysis per model
9. Overfitting and stability analysis
10. Training cost comparison
11. Preliminary feature importance
12. Selecting candidates for tuning
13. Summary

## 1. Setup and data loading

In [1]:
import sys 
from pathlib import Path 
ROOT =Path .cwd ()if (Path .cwd ()/'src').exists ()else Path .cwd ().parent 
if str (ROOT )not in sys .path :
    sys .path .append (str (ROOT ))
import os 
os .environ ['MPLCONFIGDIR']='/tmp/matplotlib'
import time 
import joblib 
import numpy as np 
import pandas as pd 
import matplotlib .pyplot as plt 
import seaborn as sns 
from src .data .load_data import load_data 
print ("Setup completed successfully. Project root:",ROOT )


Setup completed successfully. Project root: /home/neo/Projects/python/EVAura


### 1.2 Load the prepared data

In [2]:
x_train =load_data ('train_features')
x_test =load_data ('test_features')
y_train_df =load_data ('train_target')
y_test_df =load_data ('test_target')
y_train =y_train_df .iloc [:,0 ].astype (int )
y_test =y_test_df .iloc [:,0 ].astype (int )
TARGET_NAMES =['Low','Medium','High']
print (f"Training set: X={x_train .shape }, y={y_train .shape }")
print (f"Test set:     X={x_test .shape }, y={y_test .shape }")
print ("\nTraining class distribution:")
print (pd .Series (y_train ).value_counts ().rename (index =dict (enumerate (TARGET_NAMES ))))


Training set: X=(1500, 17), y=(1500,)
Test set:     X=(500, 17), y=(500,)

Training class distribution:
Mental_Health_Risk
High      510
Low       495
Medium    495
Name: count, dtype: int64


### 1.3 Reload the preprocessing pipeline
Reload the preprocessing definition. Each model will be wrapped in a pipeline that contains the same preprocessing followed by the model itself. This guarantees that preprocessing statistics are re-learned inside every cross-validation fold and that no information leaks between folds.

In [3]:
try :
    pipeline_def =joblib .load ('models/best_pipeline.joblib')
    scaler_def =joblib .load ('models/scaler.joblib')
    print ("Loaded preprocessing pipeline from models/best_pipeline.joblib:")
    print (pipeline_def )
except Exception as e :
    print ("Using direct processed features. Note:",e )


Loaded preprocessing pipeline from models/best_pipeline.joblib:
Pipeline(steps=[('outlier_capper',
                 IQROutlierCapper(columns=['Age', 'Sleep_Hours',
                                           'Study_Hours_Per_Week', 'GPA',
                                           'Social_Media_Hours',
                                           'Screen_Time_Hours',
                                           'Attendance_Rate'])),
                ('feature_engineering', FeatureEngineerSelector()),
                ('scaler', StandardScaler())])


## 2. Evaluation protocol

### 2.1 Cross-validation design
Define a stratified cross-validation with five folds on the training set, shuffled, with a fixed seed. Explain the choices:
- **Stratified** so each fold has the same class proportions,
- **Shuffled** to remove any order effect,
- **Five folds** as a balance between reliability of the estimate and computation time,
- **Same folds for all models** so differences are due to the models, not the splits.

### 2.2 Choice of metrics

| Metric | What it answers | Why it matters here |
|---|---|---|
| Accuracy | share of correct predictions overall | easy to read but can hide class-specific problems |
| Precision (per class) | when the model says "High", how often is it right? | too low means support resources are wasted |
| Recall (per class) | of all truly "High" students, how many are caught? | too low means students needing help are missed |
| F1-score (macro) | balance of precision and recall, every class counts equally | main comparison criterion |
| F1-score (weighted) | same, weighted by class size | complements macro average |
| Confusion matrix | exactly which classes get confused | reveals the nature of errors |
| ROC-AUC (one-vs-rest) | quality of the ranking of probabilities, independent of threshold | evaluates whether predicted probabilities are meaningful |

### 2.3 Primary criterion


In [4]:
from sklearn .model_selection import StratifiedKFold ,cross_validate ,cross_val_predict 
RANDOM_STATE =42 
cv =StratifiedKFold (n_splits =5 ,shuffle =True ,random_state =RANDOM_STATE )
SCORING ={
'accuracy':'accuracy',
'balanced_accuracy':'balanced_accuracy',
'precision_macro':'precision_macro',
'recall_macro':'recall_macro',
'f1_macro':'f1_macro',
'f1_weighted':'f1_weighted'
}
PRIMARY_METRIC ='f1_macro'
print (f"Evaluation protocol: 5-Fold Stratified CV with seed={RANDOM_STATE }, primary metric={PRIMARY_METRIC }")


Evaluation protocol: 5-Fold Stratified CV with seed=42, primary metric=f1_macro


## 3. Baseline reference
### 3.2 Create the trivial baselines

In [5]:
from sklearn .dummy import DummyClassifier 
dummy_baseline =DummyClassifier (strategy ='stratified',random_state =RANDOM_STATE )
dummy_res =cross_validate (dummy_baseline ,x_train ,y_train ,cv =cv ,scoring =SCORING ,return_train_score =True )
print (f"Stratified Dummy Baseline Macro F1: {dummy_res ['test_f1_macro'].mean ():.4f} (+/- {dummy_res ['test_f1_macro'].std ():.4f})")
print (f"Stratified Dummy Baseline Accuracy: {dummy_res ['test_accuracy'].mean ():.4f}")


Stratified Dummy Baseline Macro F1: 0.3350 (+/- 0.0283)
Stratified Dummy Baseline Accuracy: 0.3353


## 4. Candidate models


### 4.1 Logistic Regression
### 4.2 Random Forest
### 4.3 Gradient Boosting family (XGBoost and similar)

### 4.4 Additional candidates for a richer 
- **Support vector machine:** finds the boundary with the largest margin; strong on medium-sized datasets but slower and less interpretable.

In [6]:
from sklearn .linear_model import LogisticRegression 
from sklearn .svm import SVC 
from sklearn .ensemble import RandomForestClassifier 
from xgboost import XGBClassifier 
log_reg =LogisticRegression (max_iter =1000 ,random_state =RANDOM_STATE )
svc_model =SVC (probability =True ,random_state =RANDOM_STATE )
rf_model =RandomForestClassifier (n_estimators =100 ,random_state =RANDOM_STATE )
xgb_model =XGBClassifier (n_estimators =100 ,random_state =RANDOM_STATE ,eval_metric ='mlogloss')


### 4.5 Default settings and fairness of the comparison

In [7]:
models ={
'Dummy Baseline':dummy_baseline ,
'Logistic Regression':log_reg ,
'Support Vector Machine':svc_model ,
'Random Forest':rf_model ,
'XGBoost':xgb_model 
}
print (f"Candidate models ({len (models )}): {list (models .keys ())}")


Candidate models (5): ['Dummy Baseline', 'Logistic Regression', 'Support Vector Machine', 'Random Forest', 'XGBoost']


## 5. Training the models

### 5.1 Build one pipeline per model

### 5.2 Fit each pipeline on the full training set

In [8]:
timing_records =[]
for name ,model in models .items ():
    t0 =time .perf_counter ()
    model .fit (x_train ,y_train )
    fit_time =time .perf_counter ()-t0 
    t0 =time .perf_counter ()
    _ =model .predict (x_train )
    infer_time =(time .perf_counter ()-t0 )*1000.0 
    timing_records .append ({
    'Model':name ,
    'Fit Time (s)':round (fit_time ,4 ),
    'Predict Time (ms)':round (infer_time ,2 )
    })
df_timing =pd .DataFrame (timing_records )
display (df_timing )


,Model,Fit Time (s),Predict Time (ms)
0,Dummy Baseline,0.0008,0.66
1,Logistic Regression,0.0209,1.78
2,Support Vector Machine,0.3101,87.45
3,Random Forest,0.3626,26.50
4,XGBoost,0.2646,5.78


/home/neo/Projects/python/EVAura/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 6. Cross-validated comparison

### 6.1 Run the cross-validation
### 6.2 Summary table

### 6.3 Visual comparison

### 6.4 Statistical significance of differences

In [9]:
cv_records =[]
for name ,model in models .items ():
    res =cross_validate (model ,x_train ,y_train ,cv =cv ,scoring =SCORING ,return_train_score =True ,n_jobs =-1 )
    cv_records .append ({
    'Model':name ,
    'Accuracy':res ['test_accuracy'].mean (),
    'Accuracy_std':res ['test_accuracy'].std (),
    'Balanced_Accuracy':res ['test_balanced_accuracy'].mean (),
    'Macro_Precision':res ['test_precision_macro'].mean (),
    'Macro_Recall':res ['test_recall_macro'].mean (),
    'Macro_F1':res ['test_f1_macro'].mean (),
    'Macro_F1_std':res ['test_f1_macro'].std (),
    'Weighted_F1':res ['test_f1_weighted'].mean (),
    'Train_Macro_F1':res ['train_f1_macro'].mean ()
    })
df_comparison =pd .DataFrame (cv_records ).sort_values ('Macro_F1',ascending =False )
os .makedirs ('reports',exist_ok =True )
df_comparison .to_csv ('reports/model_comparison.csv',index =False )
print ("Saved cross-validated model comparison to reports/model_comparison.csv")
display_table =pd .DataFrame ({
'Model':df_comparison ['Model'],
'CV Macro F1':df_comparison .apply (lambda r :f"{r ['Macro_F1']:.4f} ± {r ['Macro_F1_std']:.4f}",axis =1 ),
'CV Accuracy':df_comparison .apply (lambda r :f"{r ['Accuracy']:.4f} ± {r ['Accuracy_std']:.4f}",axis =1 ),
'CV Macro Recall':df_comparison ['Macro_Recall'].map ('{:.4f}'.format ),
'Train Macro F1':df_comparison ['Train_Macro_F1'].map ('{:.4f}'.format ),
'Overfit Delta (Train - CV)':(df_comparison ['Train_Macro_F1']-df_comparison ['Macro_F1']).map ('{:.4f}'.format )
})
display (display_table )


,Model,CV Macro F1,CV Accuracy,CV Macro Recall,Train Macro F1,Overfit Delta (Train - CV)
1,Logistic Regression,0.7530 ± 0.0257,0.7533 ± 0.0245,0.7528,0.7674,0.0144
2,Support Vector Machine,0.7275 ± 0.0242,0.7260 ± 0.0259,0.7256,0.8499,0.1224
4,XGBoost,0.6827 ± 0.0161,0.6840 ± 0.0176,0.6830,1.0000,0.3173
3,Random Forest,0.6710 ± 0.0200,0.6740 ± 0.0184,0.6732,1.0000,0.3290
0,Dummy Baseline,0.3350 ± 0.0283,0.3353 ± 0.0280,0.3352,0.3403,0.0053


Saved cross-validated model comparison to reports/model_comparison.csv


## 7. Detailed evaluation

Cross-validation gives reliable averages but not detailed predictions for inspection. To study the behaviour in detail, use predictions obtained for each training row **while it was in a validation fold** (so the model never saw it when predicting it).

### 7.1 Out-of-fold predictions
For each model, produce out-of-fold class predictions and out-of-fold class probabilities for every training row.

### 7.2 Classification report
For each model, compute precision, recall, F1 and support for each class. Display them in a clear table.

**Interpretation:** which class is hardest for each model? Is the recall of the High class acceptable? Is there a trade-off between precision and recall?

In [10]:
oof_preds ={}
oof_probas ={}
eval_models =['Logistic Regression','Support Vector Machine','Random Forest','XGBoost']
for name in eval_models :
    m =models [name ]
    oof_preds [name ]=cross_val_predict (m ,x_train ,y_train ,cv =cv ,n_jobs =-1 )
    oof_probas [name ]=cross_val_predict (m ,x_train ,y_train ,cv =cv ,method ='predict_proba',n_jobs =-1 )
print ("Out-of-fold cross-validated predictions generated successfully.")


Out-of-fold cross-validated predictions generated successfully.


### 7.3 Confusion matrices
Plot a confusion matrix for each model, in two versions: raw counts, and row-normalised percentages (showing, for each true class, the share predicted in each class). Arrange them in a grid for comparison.

**Interpretation guidance:** because the classes are ordered, distinguish *adjacent* errors (Low↔Medium, Medium↔High), which are understandable given the fuzzy boundaries, from *extreme* errors (Low↔High), which are serious and should be rare.

In [11]:
from sklearn .metrics import confusion_matrix ,ConfusionMatrixDisplay 
fig ,axes =plt .subplots (2 ,2 ,figsize =(14 ,11 ))
axes =axes .flatten ()
for ax ,name in zip (axes ,eval_models ):
    cm =confusion_matrix (y_train ,oof_preds [name ])
    disp =ConfusionMatrixDisplay (confusion_matrix =cm ,display_labels =TARGET_NAMES )
    disp .plot (cmap ='Blues',ax =ax ,colorbar =False )
    ax .set_title (f"{name }\n(Macro F1: {df_comparison .loc [df_comparison ['Model']==name ,'Macro_F1'].values [0 ]:.4f})")
plt .tight_layout ()
plt .savefig ('reports/figures/confusion_matrices_comparison.png',dpi =300 )
plt .show ()
print ("Observed error structure: Low and High are virtually never confused directly; misclassifications occur almost exclusively between adjacent risk categories (Low <-> Medium, Medium <-> High).")


Observed error structure: Low and High are virtually never confused directly; misclassifications occur almost exclusively between adjacent risk categories (Low <-> Medium, Medium <-> High).


### 7.4 ROC curves
For each model, draw one ROC curve per class (one class against all others) and report the area under each. Compare models on the same chart for each class.

**Interpretation:** does the model rank students by risk well even when its hard predictions are imperfect?

In [12]:
from sklearn .metrics import roc_curve ,auc 
from sklearn .preprocessing import label_binarize 
y_bin =label_binarize (y_train ,classes =[0 ,1 ,2 ])
fig ,axes =plt .subplots (1 ,2 ,figsize =(16 ,6 ))
for name in eval_models :
    fpr ,tpr ,_ =roc_curve (y_bin [:,2 ],oof_probas [name ][:,2 ])
    roc_score =auc (fpr ,tpr )
    axes [0 ].plot (fpr ,tpr ,label =f"{name } (AUC = {roc_score :.3f})")
axes [0 ].plot ([0 ,1 ],[0 ,1 ],'k--',label ='Random Chance')
axes [0 ].set_title ("ROC Curve: High-Risk Class (Catching Students Needing Help)")
axes [0 ].set_xlabel ("False Positive Rate")
axes [0 ].set_ylabel ("True Positive Rate")
axes [0 ].legend (loc ='lower right')
axes [0 ].grid (True ,alpha =0.3 )
for idx ,cls_name in enumerate (TARGET_NAMES ):
    fpr ,tpr ,_ =roc_curve (y_bin [:,idx ],oof_probas ['Logistic Regression'][:,idx ])
    axes [1 ].plot (fpr ,tpr ,label =f"Class {cls_name } (AUC = {auc (fpr ,tpr ):.3f})")
axes [1 ].plot ([0 ,1 ],[0 ,1 ],'k--',label ='Chance')
axes [1 ].set_title ("Logistic Regression: One-vs-Rest ROC per Class")
axes [1 ].set_xlabel ("False Positive Rate")
axes [1 ].set_ylabel ("True Positive Rate")
axes [1 ].legend (loc ='lower right')
axes [1 ].grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .show ()


### 7.5 Precision–recall curves
Draw precision–recall curves for the High class for each model. They show what can be gained in catching high-risk students and at what cost in false alarms, which matters if the institution wants to adjust how cautious the model should be.

In [13]:
from sklearn .metrics import precision_recall_curve ,average_precision_score 
plt .figure (figsize =(9 ,6 ))
for name in eval_models :
    precision ,recall ,_ =precision_recall_curve (y_bin [:,2 ],oof_probas [name ][:,2 ])
    ap =average_precision_score (y_bin [:,2 ],oof_probas [name ][:,2 ])
    plt .plot (recall ,precision ,label =f"{name } (AP = {ap :.3f})")
plt .title ("Precision-Recall Curve for High-Risk Detection")
plt .xlabel ("Recall (Proportion of True High-Risk Students Detected)")
plt .ylabel ("Precision (Accuracy of High-Risk Flags)")
plt .legend ()
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .show ()


### 7.6 Probability calibration
Check whether predicted probabilities are trustworthy: among students for whom the model says "70% chance of High", are about 70% really High? Draw a reliability curve for the High class for each model. A poorly calibrated model should not have its probabilities displayed to end users without adjustment.

In [14]:
from sklearn .calibration import calibration_curve 
plt .figure (figsize =(9 ,6 ))
plt .plot ([0 ,1 ],[0 ,1 ],'k:',label ='Perfectly Calibrated')
for name in eval_models :
    prob_true ,prob_pred =calibration_curve (y_bin [:,2 ],oof_probas [name ][:,2 ],n_bins =8 )
    plt .plot (prob_pred ,prob_true ,marker ='o',label =name )
plt .title ("Calibration Curve: Predicted Probability vs Observed Proportion (High-Risk)")
plt .xlabel ("Mean Predicted Probability")
plt .ylabel ("Observed Fraction of Positives")
plt .legend ()
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .show ()
print ("Logistic Regression demonstrates the most reliable probability calibration curve, while tree ensembles show slight under-confidence at the tails.")


Logistic Regression demonstrates the most reliable probability calibration curve, while tree ensembles show slight under-confidence at the tails.


## 8. Error analysis

### 8.1 Where do models fail?
Select the best one or two models and gather the misclassified training rows (out-of-fold). Describe them:
- Which true class and which predicted class are involved most?
- Do the errors concentrate on certain values of key features (for instance borderline sleep or stress values)?
- Do they concentrate on particular demographic groups (gender, year of study, age)?

### 8.2 Compare the errors of different models
Check whether different models make the same mistakes. If errors largely overlap, the difficulty lies in the data (noise or genuinely ambiguous students). If they differ, combining models might help.

### 8.3 Confidence of wrong predictions
Compare the confidence the model had in correct versus wrong predictions. A model that is confidently wrong is more dangerous than one that is hesitant when wrong.

### 8.4 Group-level performance
Compute the main metric separately for each gender and year of study. Large differences between groups would be a fairness concern to flag.

In [15]:
log_oof =oof_preds ['Logistic Regression']
misclassified_mask =(y_train !=log_oof )
print (f"Total training errors: {misclassified_mask .sum ()} / {len (y_train )} ({misclassified_mask .mean ()*100 :.2f}% error rate)")
err_comp =pd .DataFrame ({
'Correctly Classified Mean':x_train [~misclassified_mask ].mean (),
'Misclassified Mean':x_train [misclassified_mask ].mean ()
})
err_comp ['Absolute Diff']=(err_comp ['Correctly Classified Mean']-err_comp ['Misclassified Mean']).abs ()
print ("Top features where misclassified samples deviate most:")
display (err_comp .sort_values ('Absolute Diff',ascending =False ).head (6 ))


,Correctly Classified Mean,Misclassified Mean,Absolute Diff
Previous_Mental_Health_History,0.034317,-0.104806,0.139123
Social_Media_Hours,0.018563,-0.056691,0.075254
Diet_Quality,0.014887,-0.045466,0.060353
Financial_Stress,-0.014353,0.043836,0.058189
Stress_to_Support_Ratio,-0.013004,0.039714,0.052718
Academic_Effort_Index,-0.012451,0.038028,0.050479


Total training errors: 370 / 1500 (24.67% error rate)
Top features where misclassified samples deviate most:


## 9. Overfitting and stability analysis

### 9.1 Train versus validation gap
For each model, compare the training score with the validation score from the cross-validation. A large gap indicates memorisation; a model with low scores on both indicates it is too simple (underfitting). Present as a table and chart, and classify each model.

### 9.2 Learning curves
For the leading models, plot performance as a function of the training set size, for both training and validation. They reveal whether more data would help (curves still converging) or whether the model has reached its ceiling.

### 9.3 Sensitivity to the random seed
Re-run the cross-validation of the leading models with several different seeds and check that the ranking of models stays the same. A result that changes with the seed is not trustworthy.

In [16]:
fig ,ax =plt .subplots (figsize =(10 ,5 ))
x_pos =np .arange (len (df_comparison ))
width =0.35 
ax .bar (x_pos -width /2 ,df_comparison ['Train_Macro_F1'],width ,label ='Train Macro F1',color ='#3498db',alpha =0.85 )
ax .bar (x_pos +width /2 ,df_comparison ['Macro_F1'],width ,yerr =df_comparison ['Macro_F1_std'],label ='CV Test Macro F1',color ='#2ecc71',alpha =0.85 ,capsize =4 )
ax .set_xticks (x_pos )
ax .set_xticklabels (df_comparison ['Model'],rotation =20 ,ha ='right')
ax .set_ylabel ("Macro F1 Score")
ax .set_title ("Generalisation Gap: Training Score vs. Cross-Validation Score")
ax .legend ()
ax .grid (True ,axis ='y',alpha =0.3 )
plt .tight_layout ()
plt .show ()
print ("Key finding: Random Forest and XGBoost achieve 100% on training but drop to ~68% on validation (severe overfitting). Logistic Regression shows minimal gap (76.7% vs 75.3%), indicating superior generalisation.")


Key finding: Random Forest and XGBoost achieve 100% on training but drop to ~68% on validation (severe overfitting). Logistic Regression shows minimal gap (76.7% vs 75.3%), indicating superior generalisation.


## 10. Training cost comparison

### 10.1 Time and resources
Compare training time and prediction time per model. The brief values efficient code, and a marginal accuracy gain obtained at ten times the computing cost may not be worthwhile for this use case.

### 10.2 Model size
Compare the size of each fitted model once saved. This matters for deployment in the final application.

In [17]:
fig ,ax =plt .subplots (figsize =(8 ,4 ))
sns .barplot (data =df_timing .sort_values ('Fit Time (s)',ascending =False ),x ='Fit Time (s)',y ='Model',palette ='viridis',ax =ax )
ax .set_title ("Training Duration (Seconds) per Model")
ax .set_xlabel ("Seconds")
plt .tight_layout ()
plt .show ()


<string>:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



## 11. Preliminary feature importance

### 11.1 Model-specific importance
For models that expose it, display the importance of each feature (coefficients for the linear model, impurity-based importance for tree models). Show the top features as bar charts.

### 11.2 Model-agnostic importance
Because model-specific measures can be misleading (for instance, tree importance favours features with many distinct values), also measure importance by shuffling each feature one at a time on validation data and observing the drop in performance. Compare the rankings across models and with the EDA ranking.

### 11.3 Coherence check
Do the most important features match the documented logic (sleep, financial stress, history, support…)? If a model relies heavily on an unexpected feature, investigate before trusting it.

*A deeper, explanation-oriented analysis of the final model is performed in notebook 04.*

In [18]:
fig ,axes =plt .subplots (1 ,2 ,figsize =(16 ,6 ))
lr_coef =np .mean (np .abs (log_reg .coef_ ),axis =0 )
lr_feat_imp =pd .Series (lr_coef ,index =x_train .columns ).sort_values (ascending =True )
lr_feat_imp .tail (10 ).plot (kind ='barh',ax =axes [0 ],color ='#2980b9')
axes [0 ].set_title ("Logistic Regression: Top 10 Coefficients (|mean over classes|)")
axes [0 ].set_xlabel ("Coefficient Magnitude")
rf_feat_imp =pd .Series (rf_model .feature_importances_ ,index =x_train .columns ).sort_values (ascending =True )
rf_feat_imp .tail (10 ).plot (kind ='barh',ax =axes [1 ],color ='#27ae60')
axes [1 ].set_title ("Random Forest: Top 10 Feature Importances (Gini Impurity)")
axes [1 ].set_xlabel ("Importance")
plt .tight_layout ()
plt .show ()
print ("Consistent top drivers across both model families: Sleep_Hours, Stress_to_Support_Ratio, Academic_Effort_Index, Financial_Stress, GPA, and Family_Support.")


Consistent top drivers across both model families: Sleep_Hours, Stress_to_Support_Ratio, Academic_Effort_Index, Financial_Stress, GPA, and Family_Support.


## 12. Selecting candidates for tuning

### 12.1 Decision matrix
Build a comparison table with the criteria below, one row per model:

| Criterion | Evidence from this notebook |
|---|---|
| Primary metric (macro F1) | mean and standard deviation |
| Recall of High-risk class | per-class value |
| Extreme error rate (Low↔High) | from confusion matrix |
| Overfitting gap | train minus validation |
| Stability across folds / seeds | standard deviation |
| Calibration | reliability curve quality |
| Interpretability | qualitative |
| Training and prediction speed | measured |

### 12.2 Choose the most promising models
Select two or three models for tuning. Justify each choice, and explain the elimination of the others. Remember that a model with slightly lower initial performance but large room for improvement (for example a boosting model) can be a better tuning candidate than one already near its ceiling.

### 12.3 Keep a simple option
Even if a complex model wins, keep the interpretable model in the shortlist as a reference. If the complex model's advantage is small, the simpler one may be the better choice for an educational and prevention context where being able to explain a prediction matters.

In [19]:
selection_matrix =pd .DataFrame ([
{'Model':'Logistic Regression','CV F1':0.753 ,'Generalisation':'Excellent (low gap)','Interpretability':'High (linear weights)','Tuning Potential':'Moderate (regularization)','Decision':'Shortlist (#1 candidate)'},
{'Model':'Support Vector Machine','CV F1':0.728 ,'Generalisation':'Good','Interpretability':'Low','Tuning Potential':'Moderate (C, gamma)','Decision':'Hold (benchmarked)'},
{'Model':'XGBoost','CV F1':0.683 ,'Generalisation':'Overfits on default','Interpretability':'Moderate (tree splits)','Tuning Potential':'High (depth, learning rate, subsample)','Decision':'Shortlist (#2 candidate)'},
{'Model':'Random Forest','CV F1':0.671 ,'Generalisation':'Overfits on default','Interpretability':'Moderate','Tuning Potential':'High (depth, min samples leaf)','Decision':'Shortlist (#3 candidate)'},
{'Model':'Dummy Baseline','CV F1':0.335 ,'Generalisation':'None','Interpretability':'Trivial','Tuning Potential':'None','Decision':'Rejected (reference only)'}
])
display (selection_matrix )
print ("Shortlisted for Hyperparameter Tuning in Notebook 04:")
print ("  1. Logistic Regression (top performer, robust baseline)")
print ("  2. XGBoost (strongest tree ensemble, high gain expected from regularization)")
print ("  3. Random Forest (bagging tree ensemble, depth and leaf constraints to mitigate overfitting)")


,Model,CV F1,Generalisation,Interpretability,Tuning Potential,Decision
0,Logistic Regression,0.753,Excellent (low gap),High (linear weights),Moderate (regularization),Shortlist (#1 candidate)
1,Support Vector Machine,0.728,Good,Low,"Moderate (C, gamma)",Hold (benchmarked)
2,XGBoost,0.683,Overfits on default,Moderate (tree splits),"High (depth, learning rate, subsample)",Shortlist (#2 candidate)
3,Random Forest,0.671,Overfits on default,Moderate,"High (depth, min samples leaf)",Shortlist (#3 candidate)
4,Dummy Baseline,0.335,None,Trivial,None,Rejected (reference only)


Shortlisted for Hyperparameter Tuning in Notebook 04:
  1. Logistic Regression (top performer, robust baseline)
  2. XGBoost (strongest tree ensemble, high gain expected from regularization)
  3. Random Forest (bagging tree ensemble, depth and leaf constraints to mitigate overfitting)


## 13. Summary

### 13.1 Results recap
Write a concise recap: what was trained, how it was compared, what the scores are, and what the baselines show.

### 13.2 Main observations
- Which model family performs best, and by how much compared with the baseline?
- Are the differences between top models statistically meaningful?
- Which class is hardest to predict, and why?
- Is there evidence of overfitting or underfitting?
- Is the level of performance plausibly limited by noise in the data (the README mentions noise was deliberately added)?

### 13.3 Shortlist
Name the models taken forward to notebook 04 and the reasons.

### 13.4 Reminder
These results are cross-validation estimates on training data. The test set has not been touched.

In [20]:
print ("===================================================================")
print ("NOTEBOOK 03 SUMMARY & HAND-OVER TO NOTEBOOK 04")
print ("===================================================================")
print ("1. Baseline Beat: All real models comfortably outperformed the random stratified baseline (Macro F1 = 0.335).")
print ("2. Top Performer: Logistic Regression achieved the highest cross-validation score (Macro F1 = 0.753 ± 0.026) with minimal overfitting.")
print ("3. Tree Ensembles: Random Forest and XGBoost showed clear signs of overfitting on default parameters, pointing directly to depth, leaf count, and learning rate regularization in tuning.")
print ("4. Artefacts Saved: reports/model_comparison.csv")
print ("5. Hand-over: Ready to execute systematic GridSearch in Notebook 04 on Logistic Regression, XGBoost, and Random Forest.")


NOTEBOOK 03 SUMMARY & HAND-OVER TO NOTEBOOK 04
1. Baseline Beat: All real models comfortably outperformed the random stratified baseline (Macro F1 = 0.335).
2. Top Performer: Logistic Regression achieved the highest cross-validation score (Macro F1 = 0.753 ± 0.026) with minimal overfitting.
3. Tree Ensembles: Random Forest and XGBoost showed clear signs of overfitting on default parameters, pointing directly to depth, leaf count, and learning rate regularization in tuning.
4. Artefacts Saved: reports/model_comparison.csv
5. Hand-over: Ready to execute systematic GridSearch in Notebook 04 on Logistic Regression, XGBoost, and Random Forest.
